# 使用 Microsoft Foundry 微調模型

本筆記本遵循目前的 [Microsoft Foundry 微調工作流程](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst)。 它使用指向您的 Foundry 資源 `/openai/v1/` 端點的 **OpenAI Python SDK (v1)**，因此只需更改客戶端設定，這段程式碼也能在 OpenAI 平台上運行。

> <strong>前置條件</strong>
> - 擁有 [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 專案，且具有 **Foundry Owner** 身份（執行微調與部署的必要角色）。
> - 執行 `pip install "openai>=1.0" python-dotenv`
> - 擁有一個包含 `AZURE_OPENAI_ENDPOINT` 及 `AZURE_OPENAI_API_KEY` 的 `.env` 檔案（請參考 [課程設置指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)）。
> - 目前支援的基礎模型，例如 `gpt-4.1-mini`（詳見 [可微調模型清單](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)）。

微調透過在與您的任務相關的額外範例上重新訓練基礎模型來提升表現。類似 _few-shot learning_ 和 _retrieval-augmented generation_ 的提示工程技巧，可以利用相關資料強化提示，但會受限於模型的上下文視窗大小。

透過微調，您會重新訓練模型本身（使用遠多於上下文視窗能容納的範例），並部署一個 _客製化_ 版本，推理時不再需要那麼多範例。這可提升品質、釋放上下文視窗空間，並能縮短提示，從而降低成本與延遲。底層方面，Foundry 使用 **LoRA（低秩適應）** 來達成高效訓練。

Foundry 支援三種技術：本教學使用的 **監督式微調 (SFT)**，此外還有 **DPO**（偏好對齊）及 **RFT**（強化微調）。工作流程有四個步驟：

1. 準備並上傳您的訓練 <strong>及驗證</strong> 資料。
2. 執行訓練工作以建立微調模型。
3. 監控工作、檢閱指標並選擇檢查點。
4. 部署微調模型並用於推理。

本教學示範如何用 SFT 微調 `gpt-4.1-mini`，建立一個能以打油詩回答關於元素週期表問題的聊天機器人。

---


### 步驟 1.1：準備您的資料集

讓我們建立一個聊天機器人，幫助您通過用打油詩回答有關元素的問題，來理解元素週期表。在_這個_簡單教程中，我們將僅創建一個資料集，使用一些回應範例來展示資料的預期格式。在現實應用中，您需要建立一個包含更多範例的資料集。您也可能能夠使用開放資料集（針對您的應用領域）——如果有的話，並將其重新格式化以用於微調。

由於我們專注於 `gpt-4.1-mini`，並尋求單回合回應（聊天完成），我們可以使用[這個建議格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)來創建範例，以符合 OpenAI 聊天完成的要求。如果您預期多回合對話內容，則會使用[多回合範例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，該格式包含一個 `weight` 參數以標示哪些訊息應該在微調過程中被使用（或不使用）。

我們將在這裡使用較簡單的單回合格式。資料為[jsonl 格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一條記錄，每條記錄為JSON格式物件。以下片段展示了2條記錄作範例——完整範例集（10個範例）請參見[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)檔案，我們將用於微調教程。<strong>注意：</strong>每條記錄_必須_定義在單行中（不可像格式化 JSON 檔案那樣分多行）

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在實際應用中，您將需要更大規模的範例集以達到良好效果——這其中的取捨是在回應品質與微調時間/成本之間。我們使用小規模範例集，是為了能快速完成微調以說明流程。請參考[這個 OpenAI Cookbook 範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)以獲得更複雜的微調教程。


---

### 第 1.2 步：上傳您的資料集

使用 Files API (`purpose="fine-tune"`) 將您的訓練和驗證檔案上傳至 Foundry。提供 <strong>驗證檔案</strong> 可讓 Foundry 報告驗證損失，方便您辨識過度擬合。

在執行以下程式之前，請確定您已經：
 - 安裝了 SDK：`pip install "openai>=1.0" python-dotenv`
 - 建立了包含 `AZURE_OPENAI_ENDPOINT` 和 `AZURE_OPENAI_API_KEY` 的 `.env` 檔案

程式碼會建立一個指向您的 Foundry 資源 `/openai/v1/` 端點的 OpenAI 用戶端，然後上傳兩個檔案並列印它們的 ID。


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### 步驟 2.1：使用 SDK 建立微調工作


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### 步驟 2.2：檢查工作的狀態

這裡有一些可以使用 `client.fine_tuning.jobs` API 執行的操作：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最近的 n 個微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 獲取特定工作的詳細資訊
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - 列出該工作中的事件
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - 列出可部署的檢查點（最近幾個 epoch）

當工作開始時，Foundry 首先會 _驗證訓練檔案_ 以確保資料格式正確。接著開始訓練，時間從幾分鐘到數小時不等，視模型和資料集大小而定。


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### 步驟 2.3：追蹤事件以監控進度


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### 步驟 2.4：在 Foundry 入口網站檢視狀態、指標與檢查點


你也可以在 [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) 的 **Build > Fine-tuning** 下追蹤工作。選擇你的工作以查看其狀態、訓練事件、超參數和指標。請注意以下這些指標：

- `train_loss` 和 `full_valid_loss` - 應該隨時間減少。
- `train_mean_token_accuracy` 和 `full_valid_mean_token_accuracy` - 應該增加。

如果訓練曲線與驗證曲線出現分歧，可能是過度擬合——請嘗試減少訓練世代數或使用較小的學習率乘數。下圖展示你將看到的狀態、訊息和指標面板類型（實際使用者介面依提供者而異）。

![微調工作狀態](../../../../../translated_images/zh-TW/fine-tuned-model-status.563271727bf7bfba.webp)


您也可以向下捲動視覺化面板以查看狀態訊息和指標，如下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-TW/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-TW/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 第3.1步：取得微調模型的ID

工作成功時，`response.fine_tuned_model` 會包含您的自訂模型ID（例如，`gpt-4.1-mini-2025-04-14.ft-...`）。在 Azure 上，您必須先<strong>部署</strong>該模型，然後才能調用它。


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### 步驟 3.2：部署微調後的模型

不同於 OpenAI 平台（您可以直接呼叫微調模型 ID），Microsoft Foundry 要求您先<strong>部署</strong>模型。最簡單的方法是使用 [Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)：前往 **建置 > 微調**，選擇已完成的工作，並選擇 <strong>部署</strong>。給部署命名（例如，`elements-limerick`）— 該部署名稱將用於程式碼中呼叫。

您也可以透過控制平面 REST/ARM API 程式化地部署；請參閱[部署指南](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst)。請記得，已部署的自訂模型按小時計費，且未使用的部署會在 15 天後被移除。


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### 第 3.3 步驟：在 Foundry 遊樂場測試你的微調模型

你也可以在 [Microsoft Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** 中測試已部署的模型—從模型下拉選單中選擇你的微調部署並嘗試提示。請使用你訓練時的<strong>相同系統訊息</strong>；不同的系統訊息會改變行為。

> **提示：** 將微調模型與基礎的 `gpt-4.1-mini` 並排比較。注意微調模型如何根據你的範例以打油詩格式回覆，而基礎模型僅遵循系統提示。

**這是一個刻意簡單的範例來示範流程，並非真實世界的資料集。** 在生產環境中，你會對真實資料（例如用於客服的產品目錄）進行微調，其質量差異會更加明顯—僅靠提示工程達成該質量會浪費更多的代幣。欲了解更完整範例，請參閱[OpenAI Cookbook 微調指南](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)以及[Foundry 微調教學](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst)。

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
